# 🎙️ LifeMate Master Pipeline — Interactive Testing & Deployment
### Zalo AI Challenge 2026

This master notebook integrates:
1. **ASR Ear**: Gipformer v4 fine-tune by Lộc VT (WER 10.22% / CER 9.21%).
2. **Agent Brain**: Qwen3-4B LoRA adapter by Quang Lùn (`r=16, alpha=32`).
3. **Skill Knowledge Graph**: Zero-hardcoding, dynamic distractor pruning, ReAct execution.
4. **Kaggle Stability**: Pre-configured with `torchao` conflict fix and fast 30s LoRA merge.

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Uninstall incompatible torchao & Install dependencies
!pip uninstall -y torchao
!pip install -q -U "transformers>=4.48.0" soundfile accelerate peft trl datasets sherpa-onnx huggingface_hub

In [ ]:
# 3. Clone or Update Repository
import os
REPO_URL = "https://github.com/Quandinh030106/LifeMate-Voice2Action-Agent.git"
REPO_DIR = "/kaggle/working/LifeMate-Voice2Action-Agent"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin && git reset --hard origin/main

%cd $REPO_DIR

In [ ]:
# 4. Automatically locate input directories
import glob, os, json

print("=" * 60)
print("🔍 TỰ ĐỘNG DÒ TÌM ĐƯỜNG DẪN TRÊN KAGGLE")
print("=" * 60)

# 1. Data Directory
data_paths = glob.glob("/kaggle/input/**/public_test.jsonl", recursive=True)
DATA_DIR = os.path.dirname(os.path.dirname(data_paths[0])) if data_paths else "/kaggle/input/datasets/locvt06/zaic2026-track-lifemate"
print(f"✅ Data Directory: {DATA_DIR}")
os.environ["DATA_DIR"] = DATA_DIR

# 2. LoRA Adapter Directory
lora_configs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
if lora_configs:
    LORA_DIR = os.path.dirname(lora_configs[0])
    with open(lora_configs[0], "r", encoding="utf-8") as f:
        lora_cfg = json.load(f)
    BASE_MODEL = lora_cfg.get("base_model_name_or_path", "Qwen/Qwen3-4B-Instruct-2507")
else:
    LORA_DIR = "/kaggle/input/models/quanglun/zaic-2026-lifemate-planner-qwen3-4b-lora/pytorch/epoch-1-peft-adapter/1"
    BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
print(f"✅ LoRA Adapter:   {LORA_DIR}")
print(f"   • Base Model:   {BASE_MODEL}")

# 3. Gipformer Directory
gip_files = glob.glob("/kaggle/input/**/gipformer*/**/selection.json", recursive=True) + glob.glob("/kaggle/input/**/gipformer*/**/model.pt", recursive=True)
GIP_DIR = os.path.dirname(gip_files[0]) if gip_files else "/kaggle/input/datasets/locvt06/gipformer-finetuned"
print(f"✅ Gipformer Dir:  {GIP_DIR}")
print("=" * 60)

In [ ]:
# 5. Fast-Merge LoRA Adapter to Standalone Model (~30 seconds)
import os, gc, torch

try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
    import peft.tuners.lora.torchao as peft_torchao
    peft_torchao.is_torchao_available = lambda: False
except Exception:
    pass

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_DIR = "/kaggle/working/models/lifemate-qwen3-4b-merged"

if os.path.exists(MERGED_DIR) and os.listdir(MERGED_DIR):
    print(f"✅ Đã có sẵn model merged tại: {MERGED_DIR}")
else:
    print(f"Loading Base Model {BASE_MODEL} on CPU...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map="cpu"
    )
    print(f"Nạp LoRA adapter từ {LORA_DIR}...")
    peft_model = PeftModel.from_pretrained(base_model, LORA_DIR)
    merged_model = peft_model.merge_and_unload()
    
    os.makedirs(MERGED_DIR, exist_ok=True)
    print(f"Lưu model độc lập vào {MERGED_DIR}...")
    merged_model.save_pretrained(MERGED_DIR)
    
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.save_pretrained(MERGED_DIR)
    
    del base_model, peft_model, merged_model
    gc.collect()
    print(f"🎉 Merge thành công! Model sẵn sàng tại: {MERGED_DIR}")

In [ ]:
# 6. Test nhanh 3 samples thực tế với SkillGraph và ToolEnv (Chế độ text, kết quả tức thì)
!python -u inference.py \
    --data_dir "$DATA_DIR" \
    --split public_test \
    --no_audio \
    --limit 3 \
    --llm_model "$MERGED_DIR" \
    --output "/kaggle/working/test_3_samples.jsonl"

print("\n" + "=" * 70)
print("🎯 KẾT QUẢ GỌI TOOL & STATE_HASH CỦA 3 SAMPLE ĐẦU TIÊN:")
print("=" * 70)
with open("/kaggle/working/test_3_samples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

In [ ]:
# 7. Xem thông tin selection.json của Gipformer v4
import json
sel_path = os.path.join(GIP_DIR, "selection.json")
if os.path.exists(sel_path):
    print("📖 [Thông số selection.json]:\n")
    with open(sel_path, "r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("selection.json not found in GIP_DIR")

In [ ]:
# 8. (Tùy chọn) Chạy Full Inference khi bạn sẵn sàng
# !python -u inference.py \
#     --data_dir "$DATA_DIR" \
#     --split public_test \
#     --llm_model "$MERGED_DIR" \
#     --output "/kaggle/working/submission.jsonl"